In [6]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/home/csutter/DRIVE-clean/ice/WTA/WTA_road_conditions_20220913.csv")

df.columns

df.head(4)

,Area_Name,Roadway,Description,Updated_Date,Updated_By,Action_Desc,StatusGroup_Name,Status_Name
0,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-02 12:47:52.917,stephen.fancher@dot.ny.gov,Update Area,Overall Status,Update Pending
1,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-02 12:48:20.840,stephen.fancher@dot.ny.gov,Update Area,Overall Status,Wet
2,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-02 12:48:20.840,stephen.fancher@dot.ny.gov,Update Area,Pavement Conditions,Wet Spots
3,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-02 12:48:20.840,stephen.fancher@dot.ny.gov,Update Area,Weather Conditions,Rain


In [14]:
# grab icy pavement conditions

pavement = df[df["StatusGroup_Name"]=="Pavement Conditions"]

icy = pavement[pavement["Status_Name"].isin(['Icy Spots','Icy Stretches'])]

icy["date_dt"] = pd.to_datetime(icy['Updated_Date'])
icy["date_gmt"] = icy['date_dt'].dt.tz_localize('America/New_York').dt.tz_convert('UTC')

icy["year"] = icy["date_gmt"].dt.year
icy["month"] = icy["date_gmt"].dt.month
icy["day"] = icy["date_gmt"].dt.day
icy["hour"] = icy["date_gmt"].dt.hour
icy["datetime"] = (
    icy["year"].astype(str).str.zfill(4) +
    icy["month"].astype(str).str.zfill(2) +
    icy["day"].astype(str).str.zfill(2) +
    "_" +
    icy["hour"].astype(str).str.zfill(2)+
    "00"
)
display(icy.head(6))



/tmp/tmp.RmocUWJbx6/ipykernel_2844644/1285181554.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  icy["date_dt"] = pd.to_datetime(icy['Updated_Date'])
/tmp/tmp.RmocUWJbx6/ipykernel_2844644/1285181554.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  icy["date_gmt"] = icy['date_dt'].dt.tz_localize('America/New_York').dt.tz_convert('UTC')
/tmp/tmp.RmocUWJbx6/ipykernel_2844644/1285181554.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_i

,Area_Name,Roadway,Description,Updated_Date,Updated_By,Action_Desc,StatusGroup_Name,Status_Name,date_dt,date_gmt,year,month,day,hour,datetime
14,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-02 22:42:05.720,martin.lawless@dot.ny.gov,Update Area,Pavement Conditions,Icy Spots,2022-01-02 22:42:05.720,2022-01-03 03:42:05.720000+00:00,2022,1,3,3,20220103_0300
21,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-03 02:07:26.000,stephen.fancher@dot.ny.gov,Update Area,Pavement Conditions,Icy Spots,2022-01-03 02:07:26.000,2022-01-03 07:07:26+00:00,2022,1,3,7,20220103_0700
24,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-03 05:48:55.000,stephen.fancher@dot.ny.gov,Update Area,Pavement Conditions,Icy Spots,2022-01-03 05:48:55.000,2022-01-03 10:48:55+00:00,2022,1,3,10,20220103_1000
33,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-06 05:08:22.347,stephen.fancher@dot.ny.gov,Update Area,Pavement Conditions,Icy Spots,2022-01-06 05:08:22.347,2022-01-06 10:08:22.347000+00:00,2022,1,6,10,20220106_1000
34,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-06 05:08:22.347,stephen.fancher@dot.ny.gov,Update Area,Pavement Conditions,Icy Stretches,2022-01-06 05:08:22.347,2022-01-06 10:08:22.347000+00:00,2022,1,6,10,20220106_1000
48,Albany,I-87-Q2,Northway from Rte 20 to Twin Bridges,2022-01-07 06:03:04.000,stephen.fancher@dot.ny.gov,Update Area,Pavement Conditions,Icy Spots,2022-01-07 06:03:04.000,2022-01-07 11:03:04+00:00,2022,1,7,11,20220107_1100


In [27]:
# number of unique datetimes with icy preds

print(len(np.unique(icy["datetime"])))

# grab datetimes that have the most location

df2 = icy[["Area_Name","datetime"]].groupby(["datetime"]).count()
df2 = df2.sort_values(["Area_Name"], ascending = False).reset_index()


# grab top 500 most numerous datetimes
topdates = df2[0:500]
topdates



1233


,datetime,Area_Name
0,20220109_1100,301
1,20220109_1200,289
2,20220204_0400,267
3,20220109_1300,259
4,20220218_1600,231
...,...,...
495,20220123_0600,39
496,20220218_0300,39
497,20220312_2100,39
498,20220202_1000,39


In [34]:
# save the top 500 dates out as csv for running 

datestorun = list(topdates["datetime"])
forcsv = pd.DataFrame(datestorun, columns=['date'])

forcsv

In [36]:
forcsv.to_csv("/home/csutter/DRIVE-clean/operational_runs/set4_iceWTA_20250925/dates.csv")